In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\user\Desktop\Data bricks\ML\Feature_Engineering_CR_Project.csv")

In [3]:
df.head()

,age,monthly_income,loan_amount,interest_rate,tenure_months,existing_loans_count,credit_score,dpd_last_6m,years_in_current_job,loan_to_income_ratio,emi_to_income_ratio,employment_type,region,default_flag
0,45,19093,1158834,18.36,60,1,664,58,13.5,5.057848,1.011570,Business,West,1
1,55,48019,73341,12.66,60,5,593,13,1.4,0.127278,0.025456,Unemployed,East,1
2,29,15395,938362,19.08,12,2,822,68,16.7,5.079366,5.079366,Business,West,1
3,44,132797,888330,9.24,24,4,836,45,12.8,0.557449,0.278724,Unemployed,West,0
4,34,60102,1459797,21.39,48,0,813,71,20.0,2.024055,0.506014,Self_Employed,East,1


In [4]:
df["region"].unique()

array(['West', 'East', 'South', 'North'], dtype=object)

Handle categorical values

In [5]:
df = pd.get_dummies(df,columns=["employment_type","region"],drop_first=True)

In [6]:
df.head()

,age,monthly_income,loan_amount,interest_rate,tenure_months,existing_loans_count,credit_score,dpd_last_6m,years_in_current_job,loan_to_income_ratio,emi_to_income_ratio,default_flag,employment_type_Salaried,employment_type_Self_Employed,employment_type_Unemployed,region_North,region_South,region_West
0,45,19093,1158834,18.36,60,1,664,58,13.5,5.057848,1.011570,1,False,False,False,False,False,True
1,55,48019,73341,12.66,60,5,593,13,1.4,0.127278,0.025456,1,False,False,True,False,False,False
2,29,15395,938362,19.08,12,2,822,68,16.7,5.079366,5.079366,1,False,False,False,False,False,True
3,44,132797,888330,9.24,24,4,836,45,12.8,0.557449,0.278724,0,False,False,True,False,False,True
4,34,60102,1459797,21.39,48,0,813,71,20.0,2.024055,0.506014,1,False,True,False,False,False,False


Train test split

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X = df.drop("default_flag",axis=1)
y = df["default_flag"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

Standard scalar

In [10]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler = StandardScaler()

In [12]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

Train model logestic regression

In [13]:
from sklearn.linear_model import LogisticRegression

In [75]:
model = LogisticRegression(max_iter=1000)

In [15]:
model.fit(X_train_scaled,y_train)

LogisticRegression(max_iter=1000)

Evaluate model

In [16]:
from sklearn.metrics import roc_auc_score, confusion_matrix

In [17]:
y_pred_prob = model.predict_proba(X_test_scaled)[:,1]

In [18]:
auc = roc_auc_score(y_test,y_pred_prob)

In [19]:
print(auc)

0.8561425187931212


In [20]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient":model.coef_[0]
})

In [21]:
coef_df.sort_values(by="coefficient",ascending=False)

,feature,coefficient
9,loan_to_income_ratio,2.644391
7,dpd_last_6m,0.680564
5,existing_loans_count,0.678000
10,emi_to_income_ratio,0.285523
0,age,0.183940
11,employment_type_Salaried,0.150563
2,loan_amount,0.117323
4,tenure_months,0.059429
12,employment_type_Self_Employed,0.047224
3,interest_rate,0.026144


KS Metric

In [22]:
results = pd.DataFrame({
    "actual":y_test,
    "prob": y_pred_prob
})

In [23]:
results

,actual,prob
521,1,0.994336
737,1,0.725140
740,1,0.612325
660,0,0.497698
411,0,0.055874
...,...,...
408,1,0.913864
332,1,0.378541
208,1,0.185071
613,0,0.759955


In [24]:
results = results.sort_values(by="prob",ascending=False)

In [25]:
results.head(5)

,actual,prob
382,1,1.000000
88,1,0.999994
493,1,0.999994
617,1,0.999984
635,1,0.999930


Create decile

In [26]:
results["decile"] = pd.qcut(results["prob"],10,labels=False)

In [27]:
results

,actual,prob,decile
382,1,1.000000,9
88,1,0.999994,9
493,1,0.999994,9
617,1,0.999984,9
635,1,0.999930,9
...,...,...,...
604,0,0.042096,0
948,0,0.040487,0
66,1,0.034991,0
741,0,0.013509,0


In [28]:
ks_table = results.groupby("decile").agg(
    total=("actual", "count"),
    bads=("actual", "sum")
)

ks_table["goods"] = ks_table["total"] - ks_table["bads"]

ks_table = ks_table.sort_index(ascending=False)

ks_table["cum_bads"] = ks_table["bads"].cumsum()
ks_table["cum_goods"] = ks_table["goods"].cumsum()

ks_table["cum_bads_pct"] = ks_table["cum_bads"] / ks_table["bads"].sum()
ks_table["cum_goods_pct"] = ks_table["cum_goods"] / ks_table["goods"].sum()

ks_table["ks"] = abs(ks_table["cum_bads_pct"] - ks_table["cum_goods_pct"])

ks = ks_table["ks"].max()

print("KS Statistic:", ks)
ks_table

KS Statistic: 0.5519513953248893


,total,bads,goods,cum_bads,cum_goods,cum_bads_pct,cum_goods_pct,ks
decile,,,,,,,,
9,20,20,0,20,0,0.170940,0.000000,0.170940
8,20,20,0,40,0,0.341880,0.000000,0.341880
7,20,16,4,56,4,0.478632,0.048193,0.430440
6,20,15,5,71,9,0.606838,0.108434,0.498404
5,20,13,7,84,16,0.717949,0.192771,0.525178
4,20,13,7,97,23,0.829060,0.277108,0.551951
3,20,8,12,105,35,0.897436,0.421687,0.475749
2,20,6,14,111,49,0.948718,0.590361,0.358357
1,20,5,15,116,64,0.991453,0.771084,0.220369


In [29]:
results["rank"] = results["prob"].rank(method="first")

In [30]:
results

,actual,prob,decile,rank
382,1,1.000000,9,200.0
88,1,0.999994,9,199.0
493,1,0.999994,9,198.0
617,1,0.999984,9,197.0
635,1,0.999930,9,196.0
...,...,...,...,...
604,0,0.042096,0,5.0
948,0,0.040487,0,4.0
66,1,0.034991,0,3.0
741,0,0.013509,0,2.0


Confusion matrix

In [31]:
from sklearn.metrics import confusion_matrix

In [32]:
y_pred_class = (y_pred_prob > 0.5).astype(int) 

In [33]:
cm = confusion_matrix(y_test,y_pred_class)

In [34]:
cm

array([[64, 19],
       [23, 94]], dtype=int64)

WOE (Weight of evidence)

In [35]:
df_woe = pd.DataFrame({
    "credit_score": df["credit_score"],
    "default": df["default_flag"]
})

In [36]:
df_woe["credit_bin"] = pd.qcut(
    df_woe["credit_score"],
    5,
    duplicates="drop"
)

In [37]:
df_woe.groupby("credit_bin")["default"].mean()

C:\Users\user\AppData\Local\Temp\ipykernel_8916\1608316723.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_woe.groupby("credit_bin")["default"].mean()


credit_bin
(549.999, 608.0]    0.716418
(608.0, 670.0]      0.620000
(670.0, 729.4]      0.587940
(729.4, 788.2]      0.460000
(788.2, 850.0]      0.445000
Name: default, dtype: float64

In [38]:
import numpy as np

woe_table = df_woe.groupby("credit_bin").agg(
    total=("default", "count"),
    bads=("default", "sum")
)

woe_table["goods"] = woe_table["total"] - woe_table["bads"]

woe_table["dist_goods"] = woe_table["goods"] / woe_table["goods"].sum()
woe_table["dist_bads"] = woe_table["bads"] / woe_table["bads"].sum()

# Avoid division by zero
woe_table["dist_goods"] = woe_table["dist_goods"].replace(0, 0.0001)
woe_table["dist_bads"] = woe_table["dist_bads"].replace(0, 0.0001)

woe_table["woe"] = np.log(woe_table["dist_goods"] / woe_table["dist_bads"])

woe_table["iv"] = (woe_table["dist_goods"] - woe_table["dist_bads"]) * woe_table["woe"]

woe_table

C:\Users\user\AppData\Local\Temp\ipykernel_8916\2566904543.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  woe_table = df_woe.groupby("credit_bin").agg(


,total,bads,goods,dist_goods,dist_bads,woe,iv
credit_bin,,,,,,,
"(549.999, 608.0]",201,144,57,0.131336,0.254417,-0.661212,0.081382
"(608.0, 670.0]",200,124,76,0.175115,0.219081,-0.223999,0.009848
"(670.0, 729.4]",199,117,82,0.188940,0.206714,-0.089905,0.001598
"(729.4, 788.2]",200,92,108,0.248848,0.162544,0.425892,0.036756
"(788.2, 850.0]",200,89,111,0.255760,0.157244,0.486443,0.047923


Map WOE to actual values

In [39]:
woe_dict = woe_table["woe"].to_dict()

In [40]:
woe_dict

{Interval(549.999, 608.0, closed='right'): -0.6612124876386725,
 Interval(608.0, 670.0, closed='right'): -0.22399868121592775,
 Interval(670.0, 729.4, closed='right'): -0.08990514343072493,
 Interval(729.4, 788.2, closed='right'): 0.4258921941779575,
 Interval(788.2, 850.0, closed='right'): 0.48644337568297236}

In [41]:
df_woe["credit_score_woe"] = df_woe["credit_bin"].map(woe_dict)

In [42]:
df_new = df.copy()

# Create bins again on full dataset
df_new["credit_bin"] = pd.qcut(df_new["credit_score"], 5, duplicates="drop")

df_new["credit_score_woe"] = df_new["credit_bin"].map(woe_dict)

# Drop raw score
df_new = df_new.drop(columns=["credit_score", "credit_bin"])

Retrain model

In [43]:
df_new

,age,monthly_income,loan_amount,interest_rate,tenure_months,existing_loans_count,dpd_last_6m,years_in_current_job,loan_to_income_ratio,emi_to_income_ratio,default_flag,employment_type_Salaried,employment_type_Self_Employed,employment_type_Unemployed,region_North,region_South,region_West,credit_score_woe
0,45,19093,1158834,18.36,60,1,58,13.5,5.057848,1.011570,1,False,False,False,False,False,True,-0.223999
1,55,48019,73341,12.66,60,5,13,1.4,0.127278,0.025456,1,False,False,True,False,False,False,-0.661212
2,29,15395,938362,19.08,12,2,68,16.7,5.079366,5.079366,1,False,False,False,False,False,True,0.486443
3,44,132797,888330,9.24,24,4,45,12.8,0.557449,0.278724,0,False,False,True,False,False,True,0.486443
4,34,60102,1459797,21.39,48,0,71,20.0,2.024055,0.506014,1,False,True,False,False,False,False,0.486443
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,28,117311,399720,10.47,12,3,40,9.2,0.283946,0.283946,0,False,True,False,False,False,True,-0.223999
996,31,127012,293997,17.68,36,3,70,19.7,0.192893,0.064298,0,False,False,True,True,False,False,0.486443
997,33,148626,282584,12.40,60,1,16,2.2,0.158442,0.031688,0,False,False,False,False,True,False,-0.661212
998,33,69805,602202,12.56,24,3,88,10.3,0.718910,0.359455,0,False,True,False,False,False,False,0.486443


In [44]:
from sklearn.model_selection import train_test_split

In [45]:
X = df_new.drop(columns="default_flag")
y = df_new["default_flag"]

In [46]:
X_train_v1, X_test_v1, y_train_v1, y_test_v1 = train_test_split(X,y,test_size=0.2,random_state=42)

In [47]:
from sklearn.preprocessing import StandardScaler

In [48]:
scaler = StandardScaler()

In [49]:
X_train_v1_scaled = scaler.fit_transform(X_train_v1)
X_test_v1_scaled = scaler.transform(X_test_v1)

In [50]:
from sklearn.linear_model import LogisticRegression

In [51]:
model = LogisticRegression(max_iter=1000)

In [77]:
model.fit(X_train_v1_scaled,y_train_v1)

LogisticRegression(max_iter=1000)

In [53]:
y_pred_prob_v1 = model.predict_proba(X_test_v1_scaled)[:,1]

In [54]:
from sklearn.metrics import roc_auc_score 

In [55]:
roc_score = roc_auc_score(y_test_v1,y_pred_prob_v1)

In [56]:
print(roc_score)

0.8524353825558645


There is no much improvement after WOE so will continue with XGBOOST

In [57]:
from xgboost import XGBClassifier

In [58]:
from sklearn.metrics import roc_auc_score

In [59]:
xgb = XGBClassifier(
    n_estimators = 200,
    max_depth = 4,
    learning_rate = 0.05,
    random_state = 42,
    use_label_encoder = False,
    eval_metrics = "logloss"
)

In [60]:
xgb.fit(X_train,y_train)

e:\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:15:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None,
              eval_metrics='logloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None, ...)

In [61]:
y_pred_prob_xgb = xgb.predict_proba(X_test)[:,1]

In [62]:
auc_xgb = roc_auc_score(y_test,y_pred_prob_xgb)

In [63]:
print(auc_xgb)

0.8376068376068376


In [64]:
results = pd.DataFrame({
    "actual":y_test,
    "prob": y_pred_prob_xgb
})

In [65]:
results

,actual,prob
521,1,0.989299
737,1,0.751319
740,1,0.173830
660,0,0.455167
411,0,0.027024
...,...,...
408,1,0.853718
332,1,0.495129
208,1,0.117022
613,0,0.473614


In [66]:
results = results.sort_values(by="prob",ascending=False)

In [67]:
results.head(5)

,actual,prob
617,1,0.998289
986,1,0.998025
595,1,0.997840
382,1,0.997701
513,1,0.997330


Create decile

In [68]:
results["decile"] = pd.qcut(results["prob"],10,labels=False)

In [69]:
results

,actual,prob,decile
617,1,0.998289,9
986,1,0.998025,9
595,1,0.997840,9
382,1,0.997701,9
513,1,0.997330,9
...,...,...,...
10,0,0.021322,0
621,0,0.018177,0
604,0,0.017528,0
741,0,0.016428,0


In [70]:
ks_table = results.groupby("decile").agg(
    total=("actual", "count"),
    bads=("actual", "sum")
)

ks_table["goods"] = ks_table["total"] - ks_table["bads"]

ks_table = ks_table.sort_index(ascending=False)

ks_table["cum_bads"] = ks_table["bads"].cumsum()
ks_table["cum_goods"] = ks_table["goods"].cumsum()

ks_table["cum_bads_pct"] = ks_table["cum_bads"] / ks_table["bads"].sum()
ks_table["cum_goods_pct"] = ks_table["cum_goods"] / ks_table["goods"].sum()

ks_table["ks"] = abs(ks_table["cum_bads_pct"] - ks_table["cum_goods_pct"])

ks = ks_table["ks"].max()

print("KS Statistic:", ks)
ks_table

KS Statistic: 0.5313561940067963


,total,bads,goods,cum_bads,cum_goods,cum_bads_pct,cum_goods_pct,ks
decile,,,,,,,,
9,20,20,0,20,0,0.170940,0.000000,0.170940
8,20,19,1,39,1,0.333333,0.012048,0.321285
7,20,14,6,53,7,0.452991,0.084337,0.368654
6,20,14,6,67,13,0.572650,0.156627,0.416023
5,20,15,5,82,18,0.700855,0.216867,0.483987
4,20,14,6,96,24,0.820513,0.289157,0.531356
3,20,7,13,103,37,0.880342,0.445783,0.434559
2,20,9,11,112,48,0.957265,0.578313,0.378952
1,20,4,16,116,64,0.991453,0.771084,0.220369


Cleary we can see that logistic regression is better than Xgboost

So we will proceed with scoring of logistic regression model

In [73]:
X_full = df_new.drop(columns = 'default_flag')

In [74]:
X_full_scaled = scaler.transform(X_full)

In [78]:
df_new["predicted_pd"] = model.predict_proba(X_full_scaled)[:,1]

In [80]:
df_new["predicted_pd"]

0      0.999985
1      0.586307
2      0.999986
3      0.149446
4      0.917642
         ...   
995    0.167256
996    0.043021
997    0.089273
998    0.603544
999    0.967306
Name: predicted_pd, Length: 1000, dtype: float64

In [81]:
df_new = df_new.sort_values(by = "predicted_pd",ascending=True).reset_index(drop=True)

In [85]:
df_new["rank_pct"] = (df_new.index + 1)/len(df_new)

In [86]:
df_new["rank_pct"]

0      0.001
1      0.002
2      0.003
3      0.004
4      0.005
       ...  
995    0.996
996    0.997
997    0.998
998    0.999
999    1.000
Name: rank_pct, Length: 1000, dtype: float64

In [87]:
approved = df_new[df_new["rank_pct"] <= 0.7]

In [88]:
rejected = df_new[df_new["rank_pct"] > 0.7]

In [89]:
overall_bad_rate = df_new["default_flag"].mean()
approved_bad_rate = approved["default_flag"].mean()
rejected_bad_rate = rejected["default_flag"].mean()

bad_capture_rate = rejected["default_flag"].sum() / df_new["default_flag"].sum()

print("Overall Bad Rate:", overall_bad_rate)
print("Approved Portfolio Bad Rate:", approved_bad_rate)
print("Rejected Portfolio Bad Rate:", rejected_bad_rate)
print("Bads Captured in Rejected:", bad_capture_rate)

Overall Bad Rate: 0.566
Approved Portfolio Bad Rate: 0.39571428571428574
Rejected Portfolio Bad Rate: 0.9633333333333334
Bads Captured in Rejected: 0.5106007067137809
